In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations
import scipy.stats as stats
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LinearRegression

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import plotly.express as px
#models
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import AdaBoostRegressor

In [ ]:
#importing dataset

df=pd.read_csv("/kaggle/input/datasets/simbarashemutyambizi/lthccc/Cleaned_data_final.csv")

In [ ]:
#reading dataframe
df

## Univarient analysis

I won't go into depth as this will be handled by my colleague Adnan

In [ ]:
#starting with categorical columns, separating into numeric and categorical column data
cat_df=df.select_dtypes(object)
numeric_df=df.select_dtypes([int,float])

In [ ]:
#analysis of categorical columns
#country of birth of person top ten and bottom ten
print("Top ten country of births from the data set")
print(cat_df["Country of birth of person"].value_counts()[0:10])

print(" ")

print("Bottom ten country of births from the data set")
print(cat_df["Country of birth of person"].value_counts()[260:])


India, Myanmar, Phillipeans and Liberia  and China (excludes SARs and Taiwan), Oceania and Antarctica, nfd, Eastern Europe, South Sudan, Papua New Guinea and Middle East, nfd are the top ten countries from the dataset.

We don't specifically have a bottom 5 countries in the dataset as they seem to have to the same values count.

In [ ]:
def univariate_barchart(cat_df,column):
    #accepts categorical dataframe and column to return barchart distribution
    sns.barplot(data=cat_df[column].value_counts().to_frame(),y=column,x="count")
    plt.title(f"Count of {column}")
    plt.show()


univariate_barchart(cat_df,'Years spent in Australia')

The majority of the Not specified data came from merging of the two datasets as the second dataset didn't have the column and using mode for imputation would most likely heavily skew the data.

In [ ]:
print("Age group distribution from dataset ")
print(cat_df["Age group"].value_counts())
print("-----------------------------")
print("Gender distribution from dataset ")
cat_df["Sex"].value_counts()

Each of the values from age group and sex have the same distribution of values

In [ ]:
univariate_barchart(cat_df,"Long-term health condition (LTHC)")

All conditions have the same total distribution in the dataset

In [ ]:
## total number of languages in the dataset and top ten and bottom ten most frequent languages
print("Total number of unique languages from the dataset:")
print(cat_df["Language used at home"].nunique())

print("------------------------")
print("Top ten languages used at home from the data set")
print(cat_df["Language used at home"].value_counts()[0:10])

print("--------------")

print("Bottom ten languages used at home from the data set")
print(cat_df["Language used at home"].value_counts()[238:])

We have a total of 248 languages from this dataset, with the most spoken being Spanish, French and Arabic, Creole, nfd, Dutch, Portguese, Pacific Austronesian, North European, African Languages. We technically don't have one unique low value as all of the bottm ten values have the same frequency.

Does this data also tie into the distribution of people based on the country of birth?

In [ ]:
# identifying the mode language value per Region
top_ten_language_list=cat_df["Language used at home"].value_counts().to_frame().reset_index()["Language used at home"].unique()[0:10]
top_ten_language_df=cat_df.loc[cat_df["Language used at home"].isin(top_ten_language_list)]
top_ten_language_df.groupby("Language used at home").apply(lambda x: x["Country of birth of person"].mode()[0]).to_frame()

In [ ]:
#distribution of Proficiency of spoken english values from the dataset
cat_df["Proficiency in spoken English"].value_counts()

In [ ]:
#anaylsis of region value counts
univariate_barchart(cat_df,"Region_class")

In [ ]:
#count of each subregion from the dataset
cat_df["Subregion_class"].value_counts()

Analysis of numeric data

In [ ]:
for n in ['Number of people reporting LTHC(s)', 'Population',
       'Age-specific percentage of population reporting LTHC(s)']:
    print("-----------------------------")
    plt.hist(numeric_df[n])
    plt.title(f"Distribution of numeric column {n}")
    plt.show()

Most of the numeric columns have a values close to 0, because we converted most all of the "N.P" values to Not present. Adnan has mentioned this his section and has filtered out the 0 values.

In [ ]:
non_zero_df=df.loc[df["Number of people reporting LTHC(s)"]>0]

In [ ]:
non_zero_numeric_df=numeric_df.loc[numeric_df["Number of people reporting LTHC(s)"]>0]
for n in ['Number of people reporting LTHC(s)', 'Population',
       'Age-specific percentage of population reporting LTHC(s)']:
    print("-----------------------------")
    plt.hist(non_zero_numeric_df[n])
    plt.title(f"Distribution of numeric column {n}")
    plt.show()

In [ ]:
non_zero_numeric_df.describe()

In [ ]:
#this cell and the next explain how there's a mismatch between top ten countries and top ten languages used at home
print("Number of records of people from India:")
print(len(df.loc[df["Country of birth of person"]=="India"]))
print("Number of records of people who speak French:")
print(len(df.loc[df["Language used at home"]=="French"]))

## Univariate description of the dataset

Values from columns Years spent in Australia, Age group, Gender, LTHC and English Language proficiency are of the same distribution (excluding "Not specified" values).

We have a total of 9 regions from region class, with the most occurring region being Sub-Saharan Africa

Top ten country of birth of person values were from India, Myanmar, Phillipeans and Liberia  and China (excludes SARs and Taiwan), Oceania and Antarctica, nfd, Eastern Europe, South Sudan, Papua New Guinea and Middle East, nfd.

And the top ten languages used at home where Spanish, French and Arabic, Creole, nfd, Dutch, Portguese, Pacific Austronesian, North European, African Languages. Both values obtained from the previous two sentences were based on the number of rows that had these unique values in the dataset.

In addition to that, after analyzing Top ten countries of birth of person and language used at home, we expected that there would be a match between the top one country and the top one language being from that country. However, that is not the case, even though India is the most recorded country in the dataset, the number of records from India is 3510, compared to the number of records that have french as the language used at home, with a count of 5 616.

Most of the numerical columns have values of 0, and after filtering out the 0's, we discovered that most of the data is rightly skewed, with a range of outliers. We will assess how important these values are to our target column in the next section: Bivariate analysis.





## Bivariate analysis

After re-examining the dataset in line with our previous findings from our initial bivariate analysis, we discovered we would need to conduct a different type of analysis as normal techniques like scatter plots, correlation and association metrics would not return adequate inferences of our data. 

To evaluate population health trends and project cohort-level risk, our analysis uses weighted pooled prevalence for descriptive reporting, whilst still keeping the goal of using age-specific percentages for predictive modeling. 

The reason for using Weight prevelance can be best explained with an example. For instance, lets say we are analyzing LTHC based in country of birth, age group and languge used at home for a demography (Filipino-born, aged 0–44, Tagalog-speaking) and we get the following results based on this grouping: 

Arthritis: 2,500 cases out of 10,000 people = 25% prevalence (large, reliable sample). 

Cancer: 500 cases out of 1,000 people = 50% prevalence (small, volatile sample). 

Direct analysis of raw percentages falsely suggests this group is twice as likely to contract cancer than arthritis, despite the cancer percentage being based on a sample of only 1,000 people. Hence, weighted prevalence factors in population sizes to ensure small demographic subsets do not skew statistical conclusions or create misleading risk assessments.   

Weighted prevalence establishes the true bird-view baseline of the overall population's health burden, while age-specific percentages supply the finer parameters needed to construct predictive risk models. Together, they ensure that broad descriptive findings reflect real-world population proportions without sacrificing the granularity required for accurate demographic forecasting. 

With this newly found knowledge, we developed a custom function that adds a Prevalence % column to our analysis, through three steps: 

Grouping: Groups the dataset by Data scientists selected categorical variables (such as age group, sex, or country of birth of person). 

Aggregation: Sums both the Number of people reported with LTHC and the Population within each group. 

Calculation: Divides total group cases by total group population to generate the true weighted prevalence percentage

In [ ]:
def weight_prevelance(df,col1,col2,num1,num2):
    # the function receives the two categorical columns that will be used for grouping, and also receives the two numerical columns for prevelance calculations
    weighted_df=df.groupby([col1,col2])[[num1,num2]].sum()
    weighted_df["Prevalence (%)"] = (weighted_df[num1] / weighted_df[num2]) * 100
    return weighted_df

In [ ]:
#Assessing the average number of people report to have a LTHC based on Gender and LTHC type
weight_prevelance(non_zero_df,"Sex","Long-term health condition (LTHC)","Number of people reporting LTHC(s)","Population")

In [ ]:
# converting the table aboe to grapg
sex_to_LTHC=non_zero_df.groupby(["Long-term health condition (LTHC)", "Sex"])[["Number of people reporting LTHC(s)", "Population"]].sum()
sex_to_LTHC["Prevalence (%)"] = (sex_to_LTHC["Number of people reporting LTHC(s)"] / sex_to_LTHC["Population"]) * 100
sex_to_LTHC_perc=sex_to_LTHC['Prevalence (%)'].unstack()
sex_to_LTHC_perc.plot(kind='barh', figsize=(14, 7), width=0.8, edgecolor='black')
plt.title("Percentage prevelance of people recorded with LTHC, based on gender and the specific LTHC")
plt.xlabel("Type of LTHC")
plt.ylabel("Percentage prevelance of people recorded with LTHC")
plt.show()

The graph below illustrates clear gender disparities in the prevalence of long-term health conditions (LTHCs) within the Australian population. We see that females experience substantially higher rates of Arthritis ( 9.0% compared to males at 5.7% and Persons 7%) and Mental health conditions (5.9% compared to males at 3.9% and persons 5.1).  

The graph also illustrates male-skewed conditions as well, as males are more heavily impacted by metabolic and cardiovascular illnesses, showing a higher prevalence of Diabetes (6.7% compared to females at 5.3%) and Heart disease or stroke (5.3% compared to females at 3.2%). 

For instance, we see that Arthritis has a high percentage prevalence amongst Females (9%) compared to Persons (5%) and Persons (7%).  Making this graph crucial to have in our dataset as it can be used by government and health agencies to determine which LTHC pre-dominantly occurs most amongst a specific sex.

Australian state health departments, Medicare, and organizations like the Australian Institute of Health and Welfare (AIHW) can leverage this demographic segmentation for targeted interventions like Awareness Campaigns. Rather than running generalized advertisements, health bodies can direct cardiovascular and diabetes awareness campaigns toward male-dominated media networks, workplaces, and community groups. Conversely, campaigns surrounding mental health support and arthritis management can be tailored toward female demographics as well.  

In addition to that, higher funding may be required for rheumatology and psychology services in female-dense areas, whereas cardiology and endocrinology services require scaling in populations with higher male densities. 

Considering that we have 13 conditions, each of equal distribution in our dataset, upon feedback, we narrowed down the condtions to the most common conditions in Australia from this source. https://chronichealthconditions.com.au/chronic-conditions-affecting-australians/

We will ensure our dashboard has the ability for users to choose whether the want to just view the most common conditions or all the conditions from the dataset. The most common conditions are 'Arthritis', 'Asthma', 'Diabetes','Mental health condition','Heart disease', 

I decided to return a visualization focusing on these diseases below

In [ ]:

most_common_LTHC_df=non_zero_df.loc[non_zero_df["Long-term health condition (LTHC)"].isin(['Arthritis', 'Asthma', 'Diabetes','Mental health condition','Heart disease or stroke'])]
most_common_LTHC_to_sex=most_common_LTHC_df.groupby(["Long-term health condition (LTHC)", "Sex"])[["Number of people reporting LTHC(s)", "Population"]].sum()
most_common_LTHC_to_sex["Prevalence (%)"] = (most_common_LTHC_to_sex["Number of people reporting LTHC(s)"] / most_common_LTHC_to_sex["Population"]) * 100
most_common_LTHC_to_sex_perc=most_common_LTHC_to_sex['Prevalence (%)'].unstack()

In [ ]:
most_common_LTHC_to_sex_perc.plot(kind='bar', figsize=(14, 7), width=0.8, edgecolor='black')
plt.title("Percentage prevalence of LTHC based on Sex, filtered to most common conditions in Australia")
plt.xlabel("Type of LTHC")
plt.ylabel("Percentage prevalence of people recorded with LTHC")
plt.show()

## Analysis to include in dashboard: extension of first analysis: Percentage prevelance of LTHC based on Sex and age group

In [ ]:
lthc_sex_age_group=most_common_LTHC_df.groupby(["Long-term health condition (LTHC)", "Sex","Age group"])[["Number of people reporting LTHC(s)", "Population"]].sum()
lthc_sex_age_group["Prevalence (%)"] = (lthc_sex_age_group["Number of people reporting LTHC(s)"] /lthc_sex_age_group["Population"]) * 100
lthc_sex_age_group_perc=lthc_sex_age_group['Prevalence (%)'].unstack()
lthc_sex_age_group_perc

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
lthc_sex_age_group_perc.plot(kind='barh', stacked=True, ax=ax)
plt.title("Prevalence (%) by Condition, Sex, and Age Group")
plt.ylabel("Prevalence (%)")
plt.xlabel("Condition and Sex")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()

Lets refer to arthritis amongst females in this explanation from the graph and table above
0 –44 : 0.93% 

45 – 64: 10.70% 

65 and over: 33% 

This, amongst our other results from the analysis is important as it shows that for certain groups (based on age and sex) the prevalence of a certain condition can increase as the age of that specific group increases. Without early support, this rapid growth in occurrence can strain the current resources available in the health system, hence having such data (including future data) can help government bodies to determine the average rate of group, per census and even forecast how significantly this will grow, to proactively prepare.

## Analysis to include in dashboard: Average number of people reporting with LTHC based on Sex and LTHC type

As stated in the previous analysis, the average number of women reporting with a LTHC is considerably more, compared to  men and persons, with averages in Arthirtis, Asthma, Cancer and Mental health conditions. With Diabetes (small difference between men and women), Heart disease, Heart disease or stoke and Kidney (small difference between men and women) health conditions have high average number of reportings for me.

What are the trends of number of people reporting with LTHC's based on years spent in Australia and the type of LTHC?

In [ ]:
weight_prevelance(most_common_LTHC_df,"Years spent in Australia","Long-term health condition (LTHC)","Number of people reporting LTHC(s)","Population")["Prevalence (%)"].unstack()

The table above shows a general increase in percentage prevelance the longer a person stays in Australia. This could be assessed on a deeper level, for instance adding age group to this, will take into consideration that as people age the higher the prevelance percentage

In [ ]:
most_common_without_unspecified=most_common_LTHC_df.loc[most_common_LTHC_df["Years spent in Australia"]!="Not specified"]
age_to_lthc_years_spent=most_common_without_unspecified.groupby(["Years spent in Australia","Long-term health condition (LTHC)","Age group"])[["Number of people reporting LTHC(s)", "Population"]].sum()
age_to_lthc_years_spent["Prevalence (%)"] = (age_to_lthc_years_spent["Number of people reporting LTHC(s)"] /age_to_lthc_years_spent["Population"]) * 100
age_to_lthc_years_spent_perc=age_to_lthc_years_spent['Prevalence (%)'].unstack()
age_to_lthc_years_spent_perc.plot(kind='barh', figsize=(14, 7), width=0.8, edgecolor='black')
plt.ylabel("Percentage prevelance %")
plt.title("Analysis of most common LTHC based on Age group and years spent in Australia")

plt.show()


This graph serves as a future suggestion for government and health bodies to investigate why the prevalence of LTHC rises as years spent in Australia and age group grow. A possible reason may be because of poor reporting in the early years of most migrants coming to Australia, based on fear of being deported, or for other simple reasons. 



## Second analysis to include in Dashboard: Average percentage of age specific population reporting with LTHC based on Age group and LTHC type

In [ ]:


#non_zero_dff=non_zero_df.loc[~non_zero_df["Long-term health condition (LTHC)"].isin(['Any other long-term health condition(s)','One or more long-term health condition(s)'])]
#age_to_lthc_df=non_zero_df.groupby(["Age group","Long-term health condition (LTHC)"])["Age-specific percentage of population reporting LTHC(s)"].median().unstack()
age_to_lthc_df=weight_prevelance(most_common_LTHC_df,"Age group","Long-term health condition (LTHC)","Number of people reporting LTHC(s)","Population")["Prevalence (%)"].unstack()
age_to_lthc_df.plot(kind='bar', figsize=(14, 7), width=0.8, edgecolor='black',colormap='viridis')
plt.title("Prevelence percentage of age specific population reporting with LTHC based on Age group and LTHC type")
plt.xlabel("Type of LTHC")
plt.ylabel("Prevelence percentage")
plt.show()

One of the most prominent observations we see from this graph is the increase in percentage prevalence of each health condition as the age group increases as well, supporting the common notion that the older the population gets, the higher the prevalence of long-term health conditions in that population. 

However, this graph also shows the LTHCs that are naturally pre-dominant per age group. For example, amongst the age group 0 to 44, Asthma and Mental health conditions have high LTHCs occurrences compared to the other LTCHs, and the percentage prevalence barely grows significantly (like Arthritis) as the age group increases. 

Because asthma and mental health are the predominant conditions in the 00–44 demographic, public health funding for early intervention—such as school-based asthma action plans or youth mental health services like Headspace Australia, could be directed towards educational institutions and young-adult community settings for early support and intervention. 

##  Third analysis to include in Dashboard: Analysis of Region class to Long term health condition based on number of people reporting with long term health conditions

In [ ]:
grouped_region_to_lthc=weight_prevelance(most_common_LTHC_df,"Region_class","Long-term health condition (LTHC)","Number of people reporting LTHC(s)","Population")["Prevalence (%)"].round(2).reset_index()
#index_maximum_grouped_region_to_lthc=grouped_region_to_lthc.groupby(["Region_class"])["Prevalence (%)"].idxmax()
#grouped_region_to_lthc=grouped_region_to_lthc.loc[index_maximum_grouped_region_to_lthc].reset_index(drop=True)
val_pivot = grouped_region_to_lthc.pivot(
    index="Region_class", 
    columns="Long-term health condition (LTHC)", 
    values="Prevalence (%)"
)
cond_pivot = grouped_region_to_lthc.pivot(
    index="Region_class", 
    columns="Long-term health condition (LTHC)", 
    values="Prevalence (%)"
)

# 4. Construct text label matrix (Condition Name + Percentage)
annot_matrix = cond_pivot.copy()
for r in val_pivot.index:
    for c in val_pivot.columns:
        cond = cond_pivot.loc[r, c]
        val = val_pivot.loc[r, c]
        

# 5. Plot heatmap
plt.figure(figsize=(11, 8))
sns.heatmap(
    val_pivot, 
    annot=annot_matrix, 
    fmt="", 
    cmap="YlOrRd", 
    cbar_kws={'label': 'Median Prevalence (%)'}
)

plt.title("Top LTHC & Prevalence by Region", fontsize=14, fontweight='bold')
plt.xlabel("Long-term health condition (LTHC)", fontweight='bold')
plt.ylabel("Region Class", fontweight='bold')
plt.tight_layout()
plt.show()

This heatmap serves two key analytical purposes: it highlights the leading condition within any given region, and it allows for cross-regional comparisons of specific diseases. For example, while Arthritis is the primary health condition affecting the Southern and Eastern European cohort (with a 16.26% prevalence), the heatmap reveals that the Northern European demographic actually experiences the highest overall rate of Arthritis across all nine regions. 

##  Fourth Analysis to include: Extension of previous analysis, addition Sex 

In [ ]:
# 1. Aggregate sums by Region, LTHC, and Sex
grouped_region_lthc_sex = (
    most_common_LTHC_df.groupby(["Region_class", "Long-term health condition (LTHC)", "Sex"])[
        ["Number of people reporting LTHC(s)", "Population"]
    ]
    .sum()
    .reset_index()
)

# 2. Calculate population-weighted prevalence rate (%)
grouped_region_lthc_sex["Prevalence (%)"] = (
    grouped_region_lthc_sex["Number of people reporting LTHC(s)"]
    / grouped_region_lthc_sex["Population"]
) * 100

# 3. Find the index of the highest prevalence condition PER Region and Sex
index_maximum = grouped_region_lthc_sex.groupby(["Region_class", "Sex"])[
    "Prevalence (%)"
].idxmax()

# 4. Extract top condition per Region and Sex
top_lthc_per_region = grouped_region_lthc_sex.loc[index_maximum][["Region_class", "Long-term health condition (LTHC)", "Sex", "Prevalence (%)"]]
top_lthc_per_region

In conjunction to the heatmap above, we proceeded to add Sex as another factor into the previous grouping analysis, to determine how much these LTHC percentage prevalences fluctates based on gender and region, resulting in the table below.   

We decided to limit our analysis to these finding, as adding more categorical columns to determine percentage prevalence  in the LTHC, would make the final results difficult to visualize and get a summative conclusion. Hence we will  include the findings above in our dashboard and ensure the machine learning model will be capable of capturing these variances in our final model. 